# MoE Option Backtest via Quant Orchestrator

This notebook uses the saved MoE score artifact as the signal source, then delegates option data loading and Optopsy backtesting to `quant-orchestrator`. Option chains are loaded only through `quant-warehouse` ThetaData ArcticDB download helpers.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)

_repo_start = Path.cwd().resolve()
repo_root = next(
    (candidate for candidate in (_repo_start, *_repo_start.parents) if (candidate / 'app').is_dir() and (candidate / 'notebooks').is_dir()),
    None,
)
if repo_root is None:
    raise RuntimeError(f'Could not locate optimal_trader repo root from cwd={_repo_start}')

quant_orchestrator_root = repo_root.parent / 'quant-orchestrator'
if quant_orchestrator_root.exists() and str(quant_orchestrator_root) not in sys.path:
    sys.path.insert(0, str(quant_orchestrator_root))

from quant_orchestrator.options_backtesting import (
    OptopsyBacktestSpec,
    load_score_artifact,
    run_optopsy_backtest,
    summarize_optopsy_result,
)

print(f'optimal_trader={repo_root}')
print(f'quant_orchestrator={quant_orchestrator_root}')

optimal_trader=/home/jlee153232/PycharmProjects/optimal_trader
quant_orchestrator=/home/jlee153232/PycharmProjects/quant-orchestrator


In [2]:
SCORE_ARTIFACT = repo_root / 'artifacts/moe_paper_trading/latest_scored.pkl'
TOP_K = 20
SCORE_THRESHOLD = None

# Optopsy / ThetaData settings. Set DOWNLOAD_MISSING=True to let quant-warehouse fetch missing ThetaData days.
BACKTEST_START = '2025-01-02'
BACKTEST_END = '2025-03-31'
OPTOSPY_STRATEGY = 'long_calls'
MAX_ENTRY_DTE = 60
EXIT_DTE = 30
EXIT_DTE_TOLERANCE = 3
MAX_POSITIONS = 5
DOWNLOAD_MISSING = True
ALLOW_DELTA_PROXY = True  # ThetaData ArcticDB store has real prices but may not include greeks.

# Optional explicit override. Leave empty to use scored artifact symbols; missing ThetaData is downloaded into ArcticDB when enabled.
SYMBOL_OVERRIDE = []  # Example: ['TSLA', 'NVDA', 'MSFT']
LIQUID_FALLBACK_SYMBOLS = ['TSLA', 'NVDA', 'MSFT']

selected = load_score_artifact(SCORE_ARTIFACT, top_k=TOP_K, threshold=SCORE_THRESHOLD)
display(selected.head(TOP_K))

,symbol,score,score_date,rank
0,ALLIX,0.869715,2026-06-24,1
1,HRCAX,0.857774,2026-06-24,2
2,TWCUX,0.845483,2026-06-24,3
3,FESOX,0.832770,2026-06-24,4
4,POGAX,0.820789,2026-06-24,5
5,PRUIX,0.805532,2026-06-24,6
6,RICAX,0.792501,2026-06-24,7
7,RICBX,0.780594,2026-06-24,8
8,SSEYX,0.768647,2026-06-24,9
9,FOCKX,0.751737,2026-06-24,10


In [3]:
selected_symbols = selected['symbol'].head(TOP_K).tolist()
if SYMBOL_OVERRIDE:
    symbols = [item.strip().upper() for item in SYMBOL_OVERRIDE if item.strip()]
else:
    symbols = selected_symbols

if not symbols:
    raise RuntimeError('No symbols selected for option backtest.')

symbols


Running long_calls moe_paper_trading option backtest for: ['TSLA', 'NVDA', 'MSFT']


In [4]:
spec = OptopsyBacktestSpec(
    symbols=tuple(symbols),
    start_date=BACKTEST_START,
    end_date=BACKTEST_END,
    strategy=OPTOSPY_STRATEGY,
    max_entry_dte=MAX_ENTRY_DTE,
    exit_dte=EXIT_DTE,
    exit_dte_tolerance=EXIT_DTE_TOLERANCE,
    max_positions=MAX_POSITIONS,
    download_missing=DOWNLOAD_MISSING,
    allow_delta_proxy=ALLOW_DELTA_PROXY,
)
result = run_optopsy_backtest(spec)
summary = summarize_optopsy_result(result)
display(summary)
display(result.trade_log.head(25))

,strategy,symbols,start_date,end_date,option_rows,raw_trades,closed_trades,total_trades,winning_trades,losing_trades,win_rate,total_pnl,total_return,avg_pnl,avg_win,avg_loss,max_win,max_loss,profit_factor,max_drawdown,avg_days_in_trade,sharpe_ratio,sortino_ratio,var_95,cvar_95,calmar_ratio,omega_ratio,tail_ratio
0,long_calls,"TSLA,NVDA,MSFT",2025-01-02,2025-03-31,38690,137,13,13,5,8,0.384615,-8407.0,-0.08407,-646.692308,627.0,-1442.75,2037.5,-2520.0,0.271617,-0.098381,9.923077,-2.704794,-3.043232,-0.768764,-0.836482,-1.979626,0.309391,0.0


,trade_id,underlying_symbol,entry_date,exit_date,expiration,entry_cost,exit_proceeds,quantity,multiplier,pct_change,description,exit_type,dollar_cost,dollar_proceeds,realized_pnl,days_held,cumulative_pnl,equity
0,1,MSFT,2025-01-02,2025-01-08,2025-02-07,23.200,25.850,1,100,0.114224,call 405.0,expiration,2320.0,2585.0,265.0,6,265.0,100265.0
1,2,MSFT,2025-01-08,2025-01-15,2025-02-14,30.275,31.575,1,100,0.042940,call 400.0,expiration,3027.5,3157.5,130.0,7,395.0,100395.0
2,3,TSLA,2025-01-13,2025-01-29,2025-02-28,52.500,35.975,1,100,-0.314762,call 375.0,expiration,5250.0,3597.5,-1652.5,16,-1257.5,98742.5
3,4,NVDA,2025-01-15,2025-01-22,2025-02-21,5.050,10.125,1,100,1.004950,call 141.0,expiration,505.0,1012.5,507.5,7,-750.0,99250.0
4,5,MSFT,2025-01-27,2025-02-05,2025-03-07,39.375,18.775,1,100,-0.523175,call 400.0,expiration,3937.5,1877.5,-2060.0,9,-2810.0,97190.0
5,6,TSLA,2025-02-03,2025-02-19,2025-03-21,33.275,14.050,1,100,-0.577761,call 380.0,expiration,3327.5,1405.0,-1922.5,16,-4732.5,95267.5
6,7,NVDA,2025-02-05,2025-02-12,2025-03-14,8.300,10.250,1,100,0.234940,call 129.0,expiration,830.0,1025.0,195.0,7,-4537.5,95462.5
7,8,TSLA,2025-02-12,2025-02-26,2025-03-28,34.825,9.625,1,100,-0.723618,call 320.0,expiration,3482.5,962.5,-2520.0,14,-7057.5,92942.5
8,9,NVDA,2025-02-26,2025-03-18,2025-04-17,7.675,1.255,1,100,-0.836482,call 140.0,expiration,767.5,125.5,-642.0,20,-7699.5,92300.5
9,10,TSLA,2025-02-28,2025-03-05,2025-04-04,33.675,24.050,1,100,-0.285820,call 275.0,expiration,3367.5,2405.0,-962.5,5,-8662.0,91338.0


In [5]:
output_dir = repo_root / 'artifacts' / 'quant_orchestrator_option_backtests'
output_dir.mkdir(parents=True, exist_ok=True)
prefix = 'moe_paper_trading_optopsy_' + pd.Timestamp.utcnow().strftime('%Y%m%d_%H%M%S')
summary_path = output_dir / f'{prefix}_summary.csv'
trades_path = output_dir / f'{prefix}_trades.csv'
equity_path = output_dir / f'{prefix}_equity.csv'
summary.to_csv(summary_path, index=False)
result.trade_log.to_csv(trades_path, index=False)
result.equity_curve.rename('equity').to_csv(equity_path)
{
    'summary_path': str(summary_path),
    'trades_path': str(trades_path),
    'equity_path': str(equity_path),
}


{'summary_path': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/quant_orchestrator_option_backtests/moe_paper_trading_optopsy_20260625_000112_summary.csv',
 'trades_path': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/quant_orchestrator_option_backtests/moe_paper_trading_optopsy_20260625_000112_trades.csv',
 'equity_path': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/quant_orchestrator_option_backtests/moe_paper_trading_optopsy_20260625_000112_equity.csv'}